In [2]:
import pandas as pd
import re

# 1. Load your dataset
df = pd.read_csv('master_data.csv')

# THE BRAND FILTER
# Force the data to only look at your 5 specific brands
allowed_brands = ['toyota', 'suzuki', 'honda', 'nissan', 'mitsubishi']
df = df[df['make'].str.lower().isin(allowed_brands)].copy()

# 2. Fuse Make and Model
df['make_model'] = df['make'].astype(str) + ' ' + df['model'].astype(str)

# 3. Our exact list of magic keywords
premium_keywords = 'SUPERIOR|TECH|URBAN|ZXI|CUSTOM|SAFETY|STINGRAY|TURBO|GRADE|GP5|GP1|FX'

# 4. Create a scanner to check titles
def check_premium(title):
    if pd.isna(title):
        return 0
    if re.search(premium_keywords, str(title).upper()):
        return 1
    return 0

df['has_premium_word'] = df['title'].apply(check_premium)

# 5. Group the data and calculate the stats
summary = df.groupby('make_model').agg(
    Total_Cars=('make_model', 'count'),
    Premium_Count=('has_premium_word', 'sum')
).reset_index()

# Calculate what percentage of those cars are actually premium
summary['Premium_Percentage (%)'] = (summary['Premium_Count'] / summary['Total_Cars'] * 100).round(1)

# Sort by the highest number of Premium Counts to see the real VIPs
summary = summary.sort_values(by='Premium_Count', ascending=False)

print("=== PREMIUM TRIM ANALYSIS (Filtered for Your 5 Brands) ===")
print(summary[summary['Total_Cars'] >= 5].head(40).to_string(index=False))

=== PREMIUM TRIM ANALYSIS (Filtered for Your 5 Brands) ===
                     make_model  Total_Cars  Premium_Count  Premium_Percentage (%)
        Suzuki Wagon R Stingray         180            180                   100.0
              Suzuki Wagon R FX          90             90                   100.0
       Toyota Premio G Superior          84             84                   100.0
                  Honda FIT GP5          54             54                   100.0
                  Honda Fit GP5          50             50                   100.0
                  Honda FIT GP1          46             46                   100.0
       Suzuki Wagon R FX Safety          40             40                   100.0
Suzuki Wagon R Stingray J Style          39             39                   100.0
            Toyota Axio G Grade          35             35                   100.0
            Toyota Aqua G Grade          35             35                   100.0
                  Honda Fit 

In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import re

# 1. Load data
df = pd.read_csv('master_data.csv')

# 2. Apply the VIP Guardrail directly to the TRAINING data
allowed_brands = ['toyota', 'suzuki', 'honda', 'nissan', 'mitsubishi']
df = df[df['make'].str.lower().isin(allowed_brands)].copy()

vip_base_models = ['WAGON R', 'PREMIO', 'FIT', 'AXIO', 'AQUA', 'SPACIA',
                   'CELERIO', 'SWIFT', 'ROOMY', 'YARIS', 'VITZ']

def apply_guardrail(row):
    premium_keywords = 'SUPERIOR|TECH|URBAN|ZXI|CUSTOM|SAFETY|STINGRAY|TURBO|GRADE|GP5|GP1|FX'
    model_name = str(row['model']).upper()
    title_text = str(row['title']).upper()

    # Check if it's a VIP car
    is_vip = any(vip in model_name for vip in vip_base_models)

    if is_vip and re.search(premium_keywords, title_text):
        return 1
    return 0

# Create the engineered feature before training
df['is_premium_trim'] = df.apply(apply_guardrail, axis=1)

# 3. Preparation
df['make_model'] = df['make'] + " " + df['model']
df['age'] = 2026 - df['year']
df['km_per_year'] = df['mileage'] / (df['age'] + 1)
df['is_registered'] = df['condition'].apply(lambda x: 1 if 'Registered' in str(x) else 0)

# Select Features
features = ['make_model', 'gear', 'fuelType', 'condition', 'location',
            'year', 'mileage', 'engineCc', 'age', 'km_per_year',
            'is_registered', 'is_premium_trim']

X = pd.get_dummies(df[features])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# 4. Train V7
model_v7 = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)
model_v7.fit(X_train, y_train)

# 5. Save everything
joblib.dump({'model': model_v7, 'mae': mean_absolute_error(y_test, model_v7.predict(X_test)), 'columns': X.columns}, 'car_price_model_v7.pkl')

print("Model V7 Trained & Guarded!")
print(f"Final Accuracy (R2): {r2_score(y_test, model_v7.predict(X_test)):.3f}")

Model V7 Trained & Guarded!
Final Accuracy (R2): 0.926


In [7]:
import pandas as pd
import joblib
import re

def predict_price_v7(car_details):
    """The ultimate V7 prediction engine with auto-cleaning and VIP Guardrail."""

    # 1. Load the V7 Model, Error Margin, and Exact Columns
    try:
        saved_data = joblib.load('car_price_model_v7.pkl')
        model = saved_data['model']
        mae = saved_data['mae']
        model_columns = saved_data['columns']
    except FileNotFoundError:
        return "Error: 'car_price_model_v7.pkl' not found. Make sure you trained V7!"

    # 2. 🧹 Auto-clean the user's messy text inputs!
    clean_make = str(car_details['make']).title().strip()
    clean_model = str(car_details['model']).title().strip()
    clean_gear = str(car_details['gear']).title().strip()
    clean_fuel = str(car_details['fuelType']).title().strip()
    clean_cond = str(car_details['condition']).title().strip()
    clean_loc = str(car_details['location']).title().strip()

    make_model = f"{clean_make} {clean_model}"
    is_reg = 1 if 'Registered' in clean_cond else 0

    # 3. Time & Usage Math
    current_year = 2026
    age = current_year - car_details['year']
    km_per_year = car_details['mileage'] / (age + 1)

    # 4.  Apply the VIP Guardrail
    vip_base_models = ['WAGON R', 'PREMIO', 'FIT', 'AXIO', 'AQUA', 'SPACIA',
                       'CELERIO', 'SWIFT', 'ROOMY', 'YARIS', 'VITZ']

    is_premium = 0
    # We check against the UPPERCASE version of our cleaned model to match our VIP list
    is_vip_car = any(vip in clean_model.upper() for vip in vip_base_models)

    if is_vip_car:
        premium_keywords = 'SUPERIOR|TECH|URBAN|ZXI|CUSTOM|SAFETY|STINGRAY|TURBO|GRADE|GP5|GP1|FX'
        title = str(car_details.get('title', '')).upper()
        if re.search(premium_keywords, title):
            is_premium = 1

    # 5. Build the exact data structure for the AI
    input_data = {
        'make_model': make_model,
        'gear': clean_gear,
        'fuelType': clean_fuel,
        'condition': clean_cond,
        'location': clean_loc,
        'year': car_details['year'],
        'mileage': car_details['mileage'],
        'engineCc': car_details['engineCc'],
        'age': age,
        'km_per_year': km_per_year,
        'is_registered': is_reg,
        'is_premium_trim': is_premium
    }

    input_df = pd.DataFrame([input_data])

    # 6. Format the data to perfectly match the V7 brain's memory
    input_encoded = pd.get_dummies(input_df)
    # This aligns the columns safely; any missing columns get filled with 0
    input_final = input_encoded.reindex(columns=model_columns, fill_value=0)

    # 7. Predict!
    predicted_price = model.predict(input_final)[0]

    # 8. Print Results Beautifully
    print("\n" + "="*25)
    print(f" V7 AI VALUATION: {car_details['year']} {make_model}")
    if is_premium:
        print(" Validated Premium Trim Detected!")
    elif is_vip_car and not is_premium:
        print(" Standard VIP Model (No Premium Keywords)")
    else:
        print("Standard Economy Model (Guardrail Active)")
    print("="*25)
    print(f"🎯 Target Price:      Rs. {predicted_price:,.0f}")
    print(f"📉 Great Deal (Buy):  Rs. {(predicted_price - mae):,.0f} or less")
    print(f"📈 Overpriced (Walk): Rs. {(predicted_price + mae):,.0f} or more")
    print("="*50)

    return predicted_price

In [6]:
# --- TEST IT OUT ---
my_car = {
    'make': 'Toyota',
    'model': 'Prius',
    'year': 2011,
    'mileage': 176968,
    'gear': 'Automatic',
    'fuelType': 'Petrol',
    'engineCc': 1800,
    'condition': 'Registered (Used)',
    'location': 'Bandaragama',
    'title': ''
}

predict_price_v7(my_car)


 V7 AI VALUATION: 2011 Toyota Prius
🚙 Standard Economy Model (Guardrail Active)
🎯 Target Price:      Rs. 9,135,853
📉 Great Deal (Buy):  Rs. 8,597,192 or less
📈 Overpriced (Walk): Rs. 9,674,514 or more


np.float64(9135853.105996361)

In [8]:
import pandas as pd
import joblib

print("===  AI BRAIN SCAN: FEATURE PRIORITY ===")

# 1. Load the V7 Brain
try:
    saved_data = joblib.load('car_price_model_v7.pkl')
    model = saved_data['model']
    model_columns = saved_data['columns']
except FileNotFoundError:
    print("Error: Could not find 'car_price_model_v7.pkl'")
    exit()

# 2. Extract the raw importance scores
raw_importances = model.feature_importances_

# 3. Create a DataFrame to organize them
importance_df = pd.DataFrame({
    'Feature': model_columns,
    'Importance': raw_importances
})

# 4. Group the dummy columns back into their parent categories
def group_feature_name(col_name):
    if col_name.startswith('make_model_'): return 'Make & Model'
    if col_name.startswith('location_'): return 'Location'
    if col_name.startswith('gear_'): return 'Gear (Auto/Manual)'
    if col_name.startswith('fuelType_'): return 'Fuel Type'
    if col_name.startswith('condition_'): return 'Condition'
    return col_name.title() # For Year, Mileage, Age, Enginecc, etc.

importance_df['Parent_Category'] = importance_df['Feature'].apply(group_feature_name)

# 5. Sum the importances for each parent category
grouped_importance = importance_df.groupby('Parent_Category')['Importance'].sum().reset_index()

# 6. Convert to Percentages and Sort
grouped_importance['Priority (%)'] = (grouped_importance['Importance'] * 100).round(2)
grouped_importance = grouped_importance.sort_values(by='Priority (%)', ascending=False)

# 7. Print the beautiful leaderboard
print("\nHere is exactly how much weight the AI gives to each input:")
print("-" * 50)
print(f"{'Input Variable':<25} | {'AI Priority / Weight'}")
print("-" * 50)

for index, row in grouped_importance.iterrows():
    print(f"{row['Parent_Category']:<25} | {row['Priority (%)']:>6}%")

print("-" * 50)
print("💡 Notice how things like 'Location' usually matter very little, while 'Age' and 'Model' dominate the price!")

===  AI BRAIN SCAN: FEATURE PRIORITY ===

Here is exactly how much weight the AI gives to each input:
--------------------------------------------------
Input Variable            | AI Priority / Weight
--------------------------------------------------
Gear (Auto/Manual)        |  25.94%
Make & Model              |  21.02%
Enginecc                  |  15.79%
Location                  |  10.04%
Mileage                   |    9.1%
Year                      |   7.92%
Age                       |   7.33%
Km_Per_Year               |   1.41%
Fuel Type                 |    0.9%
Is_Premium_Trim           |   0.37%
Condition                 |   0.12%
Is_Registered             |   0.06%
--------------------------------------------------
💡 Notice how things like 'Location' usually matter very little, while 'Age' and 'Model' dominate the price!


In [11]:
import pandas as pd
import joblib
import re

def predict_5_year_future(car_details, buyer_yearly_km=12000):
    """
    Predicts the 5-year future price of a car.
    buyer_yearly_km: How much the NEW buyer plans to drive every year. Default is 12,000 km (SL Average).
    """
    print("\n")
    print(" INITIALIZING 5-YEAR TIME TRAVEL...")
    print()

    # 1. Load the Brain
    try:
        saved_data = joblib.load('car_price_model_v7.pkl')
        model = saved_data['model']
        model_columns = saved_data['columns']
    except FileNotFoundError:
        return "Error: 'car_price_model_v7.pkl' not found."

    # 2. Clean Inputs & Apply Guardrail
    clean_make = str(car_details['make']).title().strip()
    clean_model = str(car_details['model']).title().strip()
    clean_gear = str(car_details['gear']).title().strip()
    clean_fuel = str(car_details['fuelType']).title().strip()
    clean_cond = str(car_details['condition']).title().strip()
    clean_loc = str(car_details['location']).title().strip()
    make_model = f"{clean_make} {clean_model}"
    is_reg = 1 if 'Registered' in clean_cond else 0

    vip_base_models = ['WAGON R', 'PREMIO', 'FIT', 'AXIO', 'AQUA', 'SPACIA', 'CELERIO', 'SWIFT', 'ROOMY', 'YARIS', 'VITZ']
    is_premium = 0
    is_vip_car = any(vip in clean_model.upper() for vip in vip_base_models)

    if is_vip_car:
        premium_keywords = 'SUPERIOR|TECH|URBAN|ZXI|CUSTOM|SAFETY|STINGRAY|TURBO|GRADE|GP5|GP1|FX'
        title = str(car_details.get('title', '')).upper()
        if re.search(premium_keywords, title):
            is_premium = 1

    # 3. Time Travel Setup
    base_year = 2026 # Current Year
    manufacture_year = car_details['year']
    base_age = base_year - manufacture_year
    base_mileage = car_details['mileage']

    print(f"🚗 Vehicle: {manufacture_year} {make_model} ({clean_gear})")
    print(f"👤 Buyer Profile: Plans to drive {buyer_yearly_km:,.0f} km per year.")
    print("-" * 65)
    print(f"{'Year':<6} | {'Simulated Age':<15} | {'Simulated Odometer':<20} | {'Predicted Value'}")
    print("-" * 65)

    future_predictions = []

    # 4. The Time Machine Loop
    for i in range(6):
        future_year = base_year + i
        simulated_age = base_age + i

        # New Logic: We add the BUYER'S expected yearly mileage, not the old owner's!
        simulated_mileage = base_mileage + (buyer_yearly_km * i)

        # We must recalculate the total average for the AI's brain because that's how it was trained
        simulated_km_per_year = simulated_mileage / (simulated_age + 1)

        input_data = {
            'make_model': make_model,
            'gear': clean_gear,
            'fuelType': clean_fuel,
            'condition': clean_cond,
            'location': clean_loc,
            'year': manufacture_year,
            'mileage': simulated_mileage,
            'engineCc': car_details['engineCc'],
            'age': simulated_age,
            'km_per_year': simulated_km_per_year, # Feeds the correctly scaled math to the AI
            'is_registered': is_reg,
            'is_premium_trim': is_premium
        }

        input_df = pd.DataFrame([input_data])
        input_encoded = pd.get_dummies(input_df)
        input_final = input_encoded.reindex(columns=model_columns, fill_value=0)

        predicted_price = model.predict(input_final)[0]
        future_predictions.append(predicted_price)

        if i == 0:
            print(f"{future_year:<6} | {simulated_age:<2} Years Old  | {simulated_mileage:,.0f} km (TODAY)   | Rs. {predicted_price:,.0f}")
        else:
            print(f"{future_year:<6} | {simulated_age:<2} Years Old  | {simulated_mileage:,.0f} km          | Rs. {predicted_price:,.0f}")

    # 5. Financial Analysis Summary
    total_loss = future_predictions[0] - future_predictions[-1]
    loss_percentage = (total_loss / future_predictions[0]) * 100

    print("-" * 65)
    print(f"📉 Total Expected Depreciation (5 Yrs): Rs. {total_loss:,.0f} (-{loss_percentage:.1f}%)")
    print("=" * 65)

# --- TEST IT WITH A CUSTOM MILEAGE ---
my_investment_car = {
    'make': 'Toyota',
    'model': 'Aqua',
    'year': 2015,
    'mileage': 85000,
    'gear': 'Automatic',
    'fuelType': 'Hybrid',
    'engineCc': 1500,
    'condition': 'Registered (Used)',
    'location': 'Colombo',
    'title': 'Toyota Aqua G Grade'
}

# Let's say the new buyer drives heavily: 15,000 km per year!
predict_5_year_future(my_investment_car, buyer_yearly_km=15000)



 INITIALIZING 5-YEAR TIME TRAVEL...

🚗 Vehicle: 2015 Toyota Aqua (Automatic)
👤 Buyer Profile: Plans to drive 15,000 km per year.
-----------------------------------------------------------------
Year   | Simulated Age   | Simulated Odometer   | Predicted Value
-----------------------------------------------------------------
2026   | 11 Years Old  | 85,000 km (TODAY)   | Rs. 8,764,153
2027   | 12 Years Old  | 100,000 km          | Rs. 8,279,236
2028   | 13 Years Old  | 115,000 km          | Rs. 8,050,907
2029   | 14 Years Old  | 130,000 km          | Rs. 7,860,693
2030   | 15 Years Old  | 145,000 km          | Rs. 7,724,246
2031   | 16 Years Old  | 160,000 km          | Rs. 7,602,483
-----------------------------------------------------------------
📉 Total Expected Depreciation (5 Yrs): Rs. 1,161,670 (-13.3%)


In [10]:
# --- LET'S TEST A CAR TO SEE ITS FUTURE! ---
my_investment_car = {
    'make': 'Toyota',
    'model': 'Prius',
    'year': 2011,
    'mileage': 176968,
    'gear': 'Automatic',
    'fuelType': 'Petrol',
    'engineCc': 1800,
    'condition': 'Registered (Used)',
    'location': 'Bandaragama',
    'title': ''
}
predict_5_year_future(my_investment_car, buyer_yearly_km=15000)




 🔮 INITIALIZING 5-YEAR TIME TRAVEL...

🚗 Vehicle: 2011 Toyota Prius (Automatic)
👤 Buyer Profile: Plans to drive 15,000 km per year.
-----------------------------------------------------------------
Year   | Simulated Age   | Simulated Odometer   | Predicted Value
-----------------------------------------------------------------
2026   | 15 Years Old  | 176,968 km (TODAY)   | Rs. 9,135,853
2027   | 16 Years Old  | 191,968 km          | Rs. 9,145,097
2028   | 17 Years Old  | 206,968 km          | Rs. 9,058,201
2029   | 18 Years Old  | 221,968 km          | Rs. 9,042,728
2030   | 19 Years Old  | 236,968 km          | Rs. 9,108,700
2031   | 20 Years Old  | 251,968 km          | Rs. 9,112,811
-----------------------------------------------------------------
📉 Total Expected Depreciation (5 Yrs): Rs. 23,042 (-0.3%)
